Setup and sanity checks

In [1]:
import pandas as pd, numpy as np
panel = pd.read_parquet("../data/panel.parquet")

print("Duplicate product-weeks:", panel.duplicated(["StockCode", "Week"]).sum())   # expect 0
print(panel["units"].describe())
print(panel.groupby("StockCode")["Week"].nunique().describe())

panel["week_id"] = panel["Week"].astype(str)

Duplicate product-weeks: 0
count    66694.000000
mean        14.585990
std         32.373601
min          1.000000
25%          2.000000
50%          5.000000
75%         13.000000
max       1153.000000
Name: units, dtype: float64
count    1700.000000
mean       39.231765
std        12.562702
min        20.000000
25%        29.000000
50%        38.000000
75%        48.000000
max        76.000000
Name: Week, dtype: float64


Naive vs. fixed-effects elasticity

In [3]:
import pyfixest as pf

# Naive regression
naive = pf.feols(
    "log_units ~ log_price",
    data=panel,
    vcov={"CRV1": "StockCode"}
)

# Fixed-effects regression
fe = pf.feols(
    "log_units ~ log_price | StockCode + week_id",
    data=panel,
    vcov={"CRV1": "StockCode"}
)

# Compare results
pf.etable([naive, fe])

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0  \
0              coef         log_price  -0.658*** <br> (0.016)   
1              coef         Intercept   2.188*** <br> (0.021)   
2                fe         StockCode                       -   
3                fe           week_id                       -   
4             stats      Observations                  66,694   
5             stats                R²                   0.215   

                        1  
0  -1.728*** <br> (0.021)  
1                          
2                       -  
3                       -  
4                  66,694  
5                    0.49  , _body=<great_tables._gt_data.Body object at 0x0000029BF8660FD0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000029BF1165110>, _spanners=Spanners([SpannerInfo(spanner_id='log_units', spanner_level=1, spanner_label=Html(text='log_units'), spanner_units=None, spanner_pattern=None, vars=['0', '1'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000029BF844A690>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000029BF0656D90>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000029BF8663D50>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000029BF27D88D0>, <great_tables._gt_data.FormatInfo object at 0x0000029BF8660B50>, <great_tables._gt_data.FormatInfo object at 0x0000029BF8661390>, <great_tables._gt_data.FormatInfo object at 0x0000029BF86A97D0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table',

Elasticity by category

In [5]:
import pandas as pd
import pyfixest as pf

rows = []

for cat, g in panel.groupby("category"):
    if g["StockCode"].nunique() < 20:
        continue

    m = pf.feols(
        "log_units ~ log_price | StockCode + week_id",
        data=g,
        vcov={"CRV1": "StockCode"}
    )

    rows.append({
        "category": cat,
        "elasticity": m.coef()["log_price"],
        "se": m.se()["log_price"],
        "products": g["StockCode"].nunique(),
        "obs": len(g)
    })

elast = pd.DataFrame(rows).sort_values("elasticity")

elast["ci_low"] = elast["elasticity"] - 1.96 * elast["se"]
elast["ci_high"] = elast["elasticity"] + 1.96 * elast["se"]

print(elast.round(3))

elast.to_csv("../data/elasticity_by_category.csv", index=False)

                   category  elasticity     se  products    obs  ci_low  \
10              Toys & Kids      -2.228  0.191        28   1211  -2.603   
9                Stationery      -1.987  0.136        67   2979  -2.254   
8             Paper & Party      -1.952  0.100       138   5687  -2.148   
1      Christmas & Seasonal      -1.862  0.165        63   1731  -2.186   
6        Lighting & Candles      -1.802  0.065       168   6756  -1.930   
0            Bags & Storage      -1.734  0.052       160   6678  -1.836   
2          Garden & Outdoor      -1.705  0.084        99   3766  -1.870   
7                     Other      -1.666  0.037       516  20430  -1.739   
3                Home Decor      -1.659  0.057       216   8413  -1.770   
5          Kitchen & Dining      -1.615  0.050       178   6852  -1.713   
4   Jewellery & Accessories      -1.351  0.132        67   2191  -1.610   

    ci_high  
10   -1.854  
9    -1.720  
8    -1.756  
1    -1.538  
6    -1.674  
0    -1.633  
2

Robustness

In [7]:
import numpy as np
import pyfixest as pf

# Control for demand using n_lines as a proxy
panel["log_lines"] = np.log(
    panel["n_lines"].where(panel["n_lines"] > 0)
)

r1 = pf.feols(
    "log_units ~ log_price + log_lines | StockCode + week_id",
    data=panel,
    vcov={"CRV1": "StockCode"}
)

# Exclude Q4 to test sensitivity to seasonality
panel_non_q4 = panel.loc[panel["is_q4"] == 0].copy()

r2 = pf.feols(
    "log_units ~ log_price | StockCode + week_id",
    data=panel_non_q4,
    vcov={"CRV1": "StockCode"}
)

pf.etable([fe, r1, r2])

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0  \
0              coef         log_price  -1.728*** <br> (0.021)   
1              coef         log_lines                           
2                fe         StockCode                       -   
3                fe           week_id                       -   
4             stats      Observations                  66,694   
5             stats                R²                    0.49   

                        1                       2  
0  -1.347*** <br> (0.019)  -1.774*** <br> (0.023)  
1   1.064*** <br> (0.007)                          
2                       -                       -  
3                       -                       -  
4                  66,694                  50,885  
5                   0.659                   0.497  , _body=<great_tables._gt_data.Body object at 0x0000029BF9818210>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000029BF8757750>, _spanners=Spanners([SpannerInfo(spanner_id='log_units', spanner_level=1, spanner_label=Html(text='log_units'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000029BF8718ED0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000029BF8718750>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000029BF8718B50>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000029BF8754710>, <great_tables._gt_data.FormatInfo object at 0x0000029BF8756C10>, <great_tables._gt_data.FormatInfo object at 0x0000029BF8755250>, <great_tables._gt_data.FormatInfo object at 0x0000029BF8755A90>, <great_tables._gt_data.FormatInfo object at 0x0000029BF8756B10>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', val

In [8]:
ppml = pf.fepois("units ~ log_price | StockCode + week_id", data=panel, vcov={"CRV1": "StockCode"})
pf.etable([fe, ppml])

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0  \
0              coef         log_price  -1.728*** <br> (0.021)   
1                fe         StockCode                       -   
2                fe           week_id                       -   
3             stats      Observations                  66,694   
4             stats                R²                    0.49   

                        1  
0  -1.921*** <br> (0.046)  
1                       -  
2                       -  
3                  66,694  
4                       -  , _body=<great_tables._gt_data.Body object at 0x0000029BF98F4590>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000029BF98F5FD0>, _spanners=Spanners([SpannerInfo(spanner_id='log_units', spanner_level=1, spanner_label=Html(text='log_units'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='units', spanner_level=1, spanner_label=Html(text='units'), spanner_units=None, spanner_pattern=None, vars=['1'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000029BF98F4110>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000029BF98F46D0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000029BF98F4310>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000029BF98F4F10>, <great_tables._gt_data.FormatInfo object at 0x0000029BF98F5690>, <great_tables._gt_data.FormatInfo object at 0x0000029BF98F4E50>, <great_tables._gt_data.FormatInfo object at 0x0000029BF98F5910>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_borde